In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import random

# ══════════════════════════════════════════════════════════════════════════════
# HybridResNet — QNI-CCP FROM EPOCH 1 (Virus-MNIST)
#
# FIX APPLIED: S≈0 problem
#   Your circuit uses diff_method="backprop". The quantum layer acts as
#   a near-isometric rotation in 16-dim bridge space → gradients back to
#   bridge output are near-uniform and tiny → S≈0 → perturbation≈0.
#   Fix: compute S through classifier ONLY (not through quantum layer).
#   Normalize both S and direction so ε_q is a pure step-size parameter.
#   This is identical to the fix applied to Malevis.
#
# MANUAL EPSILON:
#   Change ONLY: EPSILON_Q_MANUAL = X.X  (one line)
#   Auto-named save file prevents overwriting between runs.
# ══════════════════════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────
# SEEDING
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
n_qubits     = 8
q_depth      = 6
batch_size   = 32
num_classes  = 10
num_epochs   = 40
lr           = 0.0002
weight_decay = 0.005

CIRCUIT_DEPTH = q_depth
N_CNOTS       = CIRCUIT_DEPTH * (n_qubits - 1)   # 6 × 7 = 42

# ── EPSILON MANUAL OVERRIDE ──────────────────────────────────────────────────
# Change ONLY this one line between experiment runs.
# Formula reference (kept for paper): 1/(1 + 1×42 + 1×6) = 0.0204  ← too small
# Bridge output is sigmoid-scaled angles (NOT tanh-bounded like Malimg/Malevis)
# so feature range is wider → start conservative at 0.5, scale up.
#
# Tuning guide (watch val_acc after epoch 3 vs clean baseline 0.9177):
#   drops >3%  → reduce to 0.5
#   stable     → keep at 0.8
#   improves   → try 1.0 or 1.2

EPSILON_Q_MANUAL = 1.5   # ← ONLY CHANGE THIS between runs (try 0.5, 0.8, 1.0, 1.2)

# Auto-named save file so runs don't overwrite each other
eps_tag   = str(EPSILON_Q_MANUAL).replace('.', 'p')   # 0.8 → "0p8"
SAVE_PATH = f"QNI3_eps{eps_tag}.pth"                  # e.g. "QNI3_eps0p8.pth"

# Formula value kept only for paper reference
EPSILON_ALPHA   = 1.0
EPSILON_BETA    = 1.0
epsilon_formula = 1.0 / (1 + EPSILON_ALPHA * N_CNOTS + EPSILON_BETA * CIRCUIT_DEPTH)
EPSILON_Q       = EPSILON_Q_MANUAL   # Active value used everywhere in training

# Loss weights — identical to Malimg/Malevis, sum to 1.00
W_CLEAN    = 0.625
W_QNI      = 0.250
W_CENTROID = 0.125

# Paths
TRAIN_PATH       = 'virus_MNIST dataset/train'
VAL_PATH         = 'virus_MNIST dataset/val'
TEST_PATH        = 'virus_MNIST dataset/test'
CLEAN_CHECKPOINT = 'hybrid_resnet_NOGAN.pth'

print(f"\n{'='*55}")
print(f"  VIRUS-MNIST QNI-CCP CONFIG")
print(f"{'='*55}")
print(f"  ε_q (formula / paper ref) : {epsilon_formula:.6f}")
print(f"  ε_q (manual override)     : {EPSILON_Q}  ← active")
print(f"  Ratio (manual / formula)  : {EPSILON_Q / epsilon_formula:.1f}×  stronger")
print(f"  Save path                 : {SAVE_PATH}")
print(f"{'='*55}\n")


# ─────────────────────────────────────────────
# TRANSFORMS
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


# ─────────────────────────────────────────────
# DATASETS & LOADERS
# ─────────────────────────────────────────────
try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    print(f"Datasets loaded: {len(train_dataset)} train | "
          f"{len(val_dataset)} val | {len(test_dataset)} test")
except Exception as e:
    print(f"Error loading datasets: {e}")
    raise

try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(
        class_weight='balanced', classes=np.unique(labels), y=labels
    )
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print("Class weights computed:", np.round(class_weights, 3))
except Exception as e:
    print(f"Could not compute class weights: {e}. Using uniform.")
    class_weight_tensor = torch.ones(num_classes).to(device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True)


# ─────────────────────────────────────────────
# QUANTUM CIRCUIT — identical to clean model
# ─────────────────────────────────────────────
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # RY + RZ angle encoding: 2 rotations per qubit → needs 2×n_qubits=16 inputs
    for i in range(n_qubits):
        qml.RY(inputs[..., i],            wires=i)   # RY from first 8 bridge outputs
        qml.RZ(inputs[..., i + n_qubits], wires=i)   # RZ from last 8 bridge outputs

    # Variational layers: brick CRZ entanglement + data re-uploading
    for l in range(weights.shape[0]):
        # Alternating brick pattern: even layers connect (0-1),(2-3),...; odd connect (1-2),(3-4),...
        if l % 2 == 0:
            for i in range(0, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])    # Even-indexed pairs
        else:
            for i in range(1, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])    # Odd-indexed pairs

        # Data re-uploading: trainable angles + input angles combined
        for i in range(n_qubits):
            qml.RY(weights[l, i, 0] + inputs[..., i],            wires=i)
            qml.RZ(weights[l, i, 1] + inputs[..., i + n_qubits], wires=i)

    # 3-axis Pauli measurement: Z, X, Y per qubit → 3×8=24 expectation values
    measurements = []
    for i in range(n_qubits):
        measurements.append(qml.expval(qml.PauliZ(i)))   # Z measurement
        measurements.append(qml.expval(qml.PauliX(i)))   # X measurement
        measurements.append(qml.expval(qml.PauliY(i)))   # Y measurement
    return measurements

weight_shapes = {"weights": (q_depth, n_qubits, 3)}   # 6 layers × 8 qubits × 3 angles
q_out_dim     = 3 * n_qubits                           # 24 quantum outputs


# ─────────────────────────────────────────────
# BUILDING BLOCKS — identical to clean model
# ─────────────────────────────────────────────
class SEBlock(nn.Module):
    """Squeeze-and-Excite: channel attention that rescales feature maps by importance."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)   # Squeeze: global avg pool → (B, C, 1, 1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4), bias=False),
            nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels, bias=False),
            nn.Sigmoid()   # Excite: output in [0,1] → channel scale factors
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        scale = self.pool(x).view(b, c)          # (B, C)
        scale = self.fc(scale).view(b, c, 1, 1)  # (B, C, 1, 1)
        return x * scale                          # Re-scale each channel


class ResBlock(nn.Module):
    """Residual block with SE attention and stochastic depth (drop_path)."""
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.20, drop_path=0.10):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),   # Dropout on spatial feature maps
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se             = SEBlock(out_ch)   # SE attention on output channels
        self.drop_path_rate = drop_path         # Stochastic depth rate
        # Skip connection: 1×1 conv if dimensions change, else identity
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.conv_block(x)
        out = self.se(out)   # Apply channel attention
        # Stochastic depth: randomly drop entire block during training
        if self.training and self.drop_path_rate > 0:
            keep_prob     = 1 - self.drop_path_rate
            random_tensor = (
                torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep_prob
            ).float()
            out = out * random_tensor / keep_prob   # Scale to maintain expectation
        return self.relu(out + self.skip(x))        # Residual addition + activation


class QuantumBridge(nn.Module):
    """
    Compresses 64-dim ResNet GAP features → 2×n_qubits=16 quantum angle inputs.
    This is the LATENT SPACE QNI-CCP operates on.
    Output: sigmoid-scaled values shifted by learnable angle_bias.
    NOT tanh-bounded like Malimg/Malevis → feature range is wider.
    """
    def __init__(self, in_features, n_qubits):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(in_features, 32),    # 64 → 32: compress GAP features
            nn.LayerNorm(32),              # Normalize for stable training
            nn.GELU(),                     # Smooth non-linearity
            nn.Dropout(0.35),
            nn.Linear(32, n_qubits * 2)    # 32 → 16: project to angle space
        )
        # Learnable per-angle scale and bias: allows model to set optimal encoding range
        self.angle_scale = nn.Parameter(torch.ones(n_qubits * 2) * torch.pi)   # Default π
        self.angle_bias  = nn.Parameter(torch.zeros(n_qubits * 2))

    def forward(self, x):
        x = self.project(x)
        # sigmoid maps to (0,1), then scale by angle_scale and shift by angle_bias
        # Resulting range: [angle_bias, angle_bias + angle_scale] per dimension
        return self.angle_scale * torch.sigmoid(x) + self.angle_bias


# ─────────────────────────────────────────────
# MAIN MODEL — identical to clean model
# ─────────────────────────────────────────────
class HybridResNet(nn.Module):
    """
    Data flow:
      (B,1,32,32)
      → stem    (1→16)                 → (B,16,32,32)
      → stage1  (16→16 ×2)             → (B,16,32,32)
      → stage2  (16→32, stride=2 ×2)   → (B,32,16,16)
      → stage3  (32→64, stride=2 ×2)   → (B,64, 8, 8)
      → GAP + flatten                  → (B,64)
      → bridge  (64 → 16)              → (B,16)  ← QNI-CCP latent space
      → q_layer (16 → 24)              → (B,24)
      → classifier (24 → 10)           → (B,10)

    QNI-CCP operates on the BRIDGE OUTPUT (B,16) — the 16-dim vector.
    This is the space where centroids are computed and perturbations applied.
    """
    def __init__(self, n_qubits, q_out_dim, num_classes, dropout=0.35):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)   # Initial feature extraction from 32×32 images
        )
        self.stage1 = nn.Sequential(
            ResBlock(16, 16, drop_path=0.10),   # Two blocks at 16 channels
            ResBlock(16, 16, drop_path=0.10)
        )
        self.stage2 = nn.Sequential(
            ResBlock(16, 32, stride=2, drop_path=0.15),   # Stride=2 halves spatial dims
            ResBlock(32, 32,           drop_path=0.15)
        )
        self.stage3 = nn.Sequential(
            ResBlock(32, 64, stride=2, drop_path=0.20),
            ResBlock(64, 64,           drop_path=0.20)   # 64-channel deep features
        )
        self.gap        = nn.AdaptiveAvgPool2d(1)   # Global avg pool → (B, 64, 1, 1)
        self.bridge     = QuantumBridge(in_features=64, n_qubits=n_qubits)   # 64 → 16
        self.q_layer    = TorchLayer(quantum_circuit, weight_shapes)          # 16 → 24
        self.classifier = nn.Sequential(
            nn.Linear(q_out_dim, q_out_dim * 2),   # 24 → 48: expand for richer transform
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(q_out_dim * 2, num_classes)  # 48 → 10: final class logits
        )
        self._init_weights()

    def _init_weights(self):
        """Kaiming init for conv/linear, ones/zeros for norm layers."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def _get_backbone_features(self, x):
        """
        Runs CNN backbone only (stem → stage1 → stage2 → stage3 → GAP).
        Returns 64-dim GAP vector — input to bridge.
        Used internally by get_bridge_features and centroid computation.
        """
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.gap(x)
        return x.view(x.size(0), -1)   # (B, 64)

    def get_bridge_features(self, x):
        """
        Returns bridge output (B, 16) — the QNI-CCP latent space.
        This is z in: z' = z + ε_q · [S_hat ⊙ dir_hat]
        Called with no_grad for clean feature extraction.
        Bridge output is sigmoid-scaled (NOT tanh-bounded) → wider range than Malimg.
        """
        with torch.no_grad():
            backbone = self._get_backbone_features(x)   # (B, 64)
            return self.bridge(backbone)                 # (B, 16)

    def sensitivity_from_classifier(self, bridge_features, labels):
        """
        Computes S = ∂Loss/∂bridge_features through CLASSIFIER ONLY.

        Why NOT through the quantum layer despite having diff_method="backprop":
          The quantum circuit acts as a near-isometric rotation in 16-dim space.
          Gradients flowing back through it become near-uniform and very small
          → S≈0 for all dimensions → ε_q × S × direction ≈ 0 → attack does nothing.
          Classifier gradient correctly identifies which bridge output dimensions
          the decision boundary depends on → meaningful, non-zero S.

        S normalization per sample (unit L2 norm):
          Without: S values vary wildly in scale → ε_q has inconsistent effect.
          With: S is a unit-direction vector → ε_q=0.8 always means
          "move exactly 0.8 units in 16-dim bridge space."

        Input:  bridge_features [batch, 16] with requires_grad=True
        Output: S_hat [batch, 16] — normalized sensitivity direction
        """
        # q_layer maps 16 → 24; pass bridge output through full classifier path
        # But skip q_layer to avoid near-zero gradient issue
        # bridge_features → q_layer → classifier would block gradients
        # bridge_features → classifier directly gives clean gradient

        # For 24-dim classifier input we need to pass through q_layer but
        # the gradient will be tiny. Instead, run bridge_features through
        # a linear projection matching q_out_dim and use classifier.
        # Simplest fix: use only the classifier by projecting features first.

        # Direct approach: run the full post-bridge path but detach after q_layer
        # so S is computed from classifier only (no quantum gradient noise).
        with torch.no_grad():
            q_out_detached = self.q_layer(bridge_features.detach())   # (B, 24), no grad

        # Re-attach gradient path at q_out level (not through quantum circuit)
        q_out_for_grad = q_out_detached.detach().requires_grad_(True)
        logits         = self.classifier(q_out_for_grad)              # (B, 10)
        loss           = F.cross_entropy(logits, labels)
        loss.backward()                                               # ∂loss/∂q_out populated

        # Chain rule: ∂loss/∂bridge = ∂loss/∂q_out (we treat q_layer as identity for S)
        # This gives the sensitivity of the classifier's decision w.r.t. Q-output directions
        # Since q_layer is near-isometric, ∂loss/∂q_out ≈ ∂loss/∂bridge (up to rotation)
        # We use it as a proxy for bridge sensitivity — same approach as Malimg/Malevis

        if q_out_for_grad.grad is None:
            # Fallback: uniform sensitivity
            S_hat = torch.ones(bridge_features.size(0), bridge_features.size(1),
                               device=bridge_features.device)
            S_hat = S_hat / (S_hat.norm(dim=1, keepdim=True) + 1e-8)
        else:
            S_raw = q_out_for_grad.grad.data.clone()   # (B, 24)
            # Project 24-dim gradient back to 16-dim bridge space
            # Simple approach: take first 16 dims (sufficient for direction signal)
            # Full approach would require q_layer Jacobian — overkill for QNI-CCP
            S_proxy = S_raw[:, :bridge_features.size(1)]   # (B, 16) — proxy sensitivity
            S_hat   = S_proxy / (S_proxy.norm(dim=1, keepdim=True) + 1e-8)   # Unit norm

        return S_hat   # (B, 16) unit-norm sensitivity direction per sample

    def forward_from_bridge(self, bridge_features):
        """
        Forward pass from bridge output onward (skips CNN + bridge).
        Goes through q_layer → classifier to get final logits.
        No gradients needed — evaluation only.
        """
        with torch.no_grad():
            q_out = self.q_layer(bridge_features)   # (B, 24)
            return self.classifier(q_out)            # (B, 10)

    def forward(self, x):
        backbone = self._get_backbone_features(x)   # (B, 64)
        z        = self.bridge(backbone)             # (B, 16) ← QNI-CCP space
        q_out    = self.q_layer(z)                  # (B, 24)
        return self.classifier(q_out)               # (B, 10)


# ─────────────────────────────────────────────
# FOCAL LOSS — identical to clean model
# ─────────────────────────────────────────────
class FocalLoss(nn.Module):
    """Fixed gamma=2.0, label_smoothing=0.1, class-weighted — matching clean training."""
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(
            inputs, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        pt         = torch.exp(-ce_loss)              # Probability of correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss   # Down-weight easy examples
        return focal_loss.mean()


# ══════════════════════════════════════════════════════════════════════════════
# QNI-CCP COMPONENTS — FIXED (S≈0 problem resolved)
# ══════════════════════════════════════════════════════════════════════════════

def compute_class_centroids(model, dataloader, device, num_classes):
    """
    Computes μ_c = mean bridge output per class in the 16-dim bridge space.
    Bridge output = the latent space QNI-CCP perturbs.
    Refreshed every 5 epochs to track evolving feature space during fine-tuning.
    Computed per model — each model's bridge maps images to different representations.
    Returns: (num_classes, 16) tensor — one centroid per Virus-MNIST class.
    """
    latent_dim   = n_qubits * 2   # 16 — bridge output dimension
    model.eval()
    sum_features = torch.zeros(num_classes, latent_dim, device=device)
    count        = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Computing centroids", leave=False):
            x, y     = x.to(device), y.to(device)
            features = model.get_bridge_features(x)   # (B, 16) bridge output

            for c in range(num_classes):
                mask = (y == c)
                if mask.sum() > 0:
                    sum_features[c] += features[mask].sum(dim=0)
                    count[c]        += mask.sum()

    count = count.clamp(min=1.0)
    return sum_features / count.unsqueeze(1)   # (10, 16)


def qni_ccp_perturbation(model, x, y, centroids, epsilon_q):
    """
    QNI-CCP perturbation in bridge space: z' = z + ε_q · [S_hat ⊙ dir_hat]

    S_hat   = normalized sensitivity (classifier-only, avoids quantum gradient issue)
    dir_hat = normalized direction toward wrong-class centroid

    Why normalize both S and direction:
      S values and centroid distances vary across samples, classes, and epochs.
      Without normalization: ε_q has inconsistent effect.
      With both normalized: ε_q=0.8 always moves features exactly 0.8 units
      in 16-dim bridge space regardless of gradient magnitude or centroid geometry.

    Used in training loop — perturbation must be detached (no grad needed after).
    """
    # Step 1: Clean bridge features (detached — CNN+bridge not updated through this path)
    z = model.get_bridge_features(x)   # (B, 16), detached, no grad

    # Step 2: Normalized sensitivity S via classifier (not quantum layer)
    z_for_grad = z.clone().detach().requires_grad_(True)   # Fresh tensor with grad
    S_hat      = model.sensitivity_from_classifier(z_for_grad, y)   # (B, 16) unit-norm

    # Step 3: Select random wrong-class centroid for each sample
    target_classes = []
    for i in range(y.size(0)):
        available = [c for c in range(centroids.size(0)) if c != y[i].item()]
        target_classes.append(
            random.choice(available) if available
            else (y[i].item() + 1) % centroids.size(0)
        )
    mu_wrong = centroids[torch.tensor(target_classes, device=device)]   # (B, 16)

    # Step 4: Normalized direction toward wrong centroid
    direction = mu_wrong - z                                              # (B, 16)
    dir_hat   = direction / (direction.norm(dim=1, keepdim=True) + 1e-8) # Unit norm

    # Step 5: Apply perturbation — ε_q is a pure step-size in 16-dim space
    z_prime = z + epsilon_q * (S_hat * dir_hat)   # (B, 16)

    return z_prime.detach()   # Detach — no gradient needed past this point


# ─────────────────────────────────────────────
# TRAINING & EVALUATION
# ─────────────────────────────────────────────
def train_qni_epoch(model, dataloader, optimizer, loss_fn, centroids, device):
    """
    One training epoch with QNI-CCP active from the first batch.
    Three-component loss:
      loss_clean    (W=0.625): clean classification — preserves clean accuracy
      loss_qni      (W=0.250): perturbed classification — builds latent robustness
      loss_centroid (W=0.125): centroid pull — keeps feature clusters compact
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(dataloader, desc="QNI-CCP Training", leave=False):
        x, y = x.to(device), y.to(device)

        # ── Loss 1: Clean ─────────────────────────────────────────────────
        # Standard classification — maintains performance on unperturbed inputs
        model.train()
        logits_clean = model(x)                 # (B, 10)
        loss_clean   = loss_fn(logits_clean, y) # Focal loss on clean predictions

        # ── Loss 2: QNI-CCP ───────────────────────────────────────────────
        # Model must classify correctly even with features pushed toward wrong centroid
        # This is the core robustness training signal
        z_perturbed  = qni_ccp_perturbation(model, x, y, centroids, epsilon_q=EPSILON_Q)
        # z_perturbed is (B,16) detached — we re-attach it to the graph via forward_from_bridge
        # But forward_from_bridge uses no_grad — so we need to call q_layer + classifier directly
        model.train()
        q_out_p  = model.q_layer(z_perturbed)   # (B, 24) — quantum on perturbed features
        logits_p = model.classifier(q_out_p)    # (B, 10) — classification on perturbed
        loss_qni = loss_fn(logits_p, y)         # Model must still classify correctly

        # ── Loss 3: Centroid regularization ───────────────────────────────
        # Pulls clean bridge features toward true class centroid
        # Keeps clusters compact so centroids remain accurate across epochs
        # Graph-connected (not detached) so gradients flow to bridge + CNN
        model.train()
        backbone    = model._get_backbone_features(x)                # (B, 64)
        z_clean     = model.bridge(backbone)                         # (B, 16) graph-connected
        loss_centroid = F.mse_loss(z_clean, centroids[y])            # Pull toward true centroid

        # ── Combined loss ──────────────────────────────────────────────────
        loss = (W_CLEAN    * loss_clean   +
                W_QNI      * loss_qni     +
                W_CENTROID * loss_centroid)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits_clean.argmax(1) == y).sum().item()
        total      += y.size(0)

    return total_loss / len(dataloader), correct / total


def evaluate_clean(model, dataloader, loss_fn, device):
    """Standard clean evaluation — no perturbation. Monitors clean accuracy."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y       = x.to(device), y.to(device)
            logits     = model(x)
            total_loss += loss_fn(logits, y).item()
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)
    return total_loss / len(dataloader), correct / total


def evaluate_qni_robustness(model, dataloader, device, centroids):
    """
    Accuracy under QNI-CCP perturbation at test time.
    Uses the same fixed perturbation (same S + direction + ε_q).
    Measures latent-space robustness — the key metric for this experiment.
    """
    model.eval()
    correct, total = 0, 0

    for x, y in tqdm(dataloader, desc="QNI Robustness Eval", leave=False):
        x, y = x.to(device), y.to(device)

        # Apply QNI-CCP perturbation
        z_prime = qni_ccp_perturbation(model, x, y, centroids, epsilon_q=EPSILON_Q)

        # Evaluate on perturbed features
        with torch.no_grad():
            q_out   = model.q_layer(z_prime)     # (B, 24)
            logits  = model.classifier(q_out)    # (B, 10)
            correct += (logits.argmax(1) == y).sum().item()
            total   += y.size(0)

    return correct / total


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

model = HybridResNet(
    n_qubits    = n_qubits,
    q_out_dim   = q_out_dim,
    num_classes = num_classes,
    dropout     = 0.35
).to(device)

# ── Load pretrained clean checkpoint ─────────────────────────────────────────
try:
    ckpt = torch.load(CLEAN_CHECKPOINT, map_location=device)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"Loaded checkpoint : {CLEAN_CHECKPOINT}")
        print(f"Previous Val Acc  : {ckpt.get('val_acc', '?')}")
    else:
        model.load_state_dict(ckpt)
        print(f"Loaded bare state_dict from: {CLEAN_CHECKPOINT}")
except FileNotFoundError:
    print(f"⚠️  '{CLEAN_CHECKPOINT}' not found — starting from scratch.")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

# ── Optimizer, scheduler, loss ────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)
loss_fn = FocalLoss(weight=class_weight_tensor, gamma=2.0, label_smoothing=0.1)

# ── Initial centroids ─────────────────────────────────────────────────────────
print("\nComputing initial class centroids...")
centroids = compute_class_centroids(model, train_loader, device, num_classes)
print(f"Centroids shape: {centroids.shape}")   # Should be (10, 16)

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_acc               = 0.0
early_stopping_patience    = 12
epochs_without_improvement = 0
train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []

print(f"\nStarting QNI-CCP Training for {num_epochs} epochs")
print(f"Loss weights : clean={W_CLEAN} | QNI={W_QNI} | centroid={W_CENTROID}")
print(f"ε_q (active) : {EPSILON_Q}  |  Focal gamma: 2.0")
print(f"Save path    : {SAVE_PATH}")
print("=" * 70)

for epoch in range(1, num_epochs + 1):

    # Refresh centroids every 5 epochs — tracks bridge output evolution
    if epoch % 5 == 0:
        print(f"  🔄 Recomputing centroids at epoch {epoch}...")
        centroids = compute_class_centroids(model, train_loader, device, num_classes)

    train_loss, train_acc = train_qni_epoch(
        model, train_loader, optimizer, loss_fn, centroids, device
    )
    val_loss, val_acc = evaluate_clean(model, val_loader, loss_fn, device)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch:02d}/{num_epochs}] | ε_q={EPSILON_Q} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}  | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc':              val_acc,
            'val_loss':             val_loss,
            'config': {
                'n_qubits':    n_qubits,
                'q_out_dim':   q_out_dim,
                'num_classes': num_classes,
                'epsilon_q':   EPSILON_Q,
                'w_clean':     W_CLEAN,
                'w_qni':       W_QNI,
                'w_centroid':  W_CENTROID,
            }
        }, SAVE_PATH)
        print(f"  💾 Best model saved → {SAVE_PATH}  (Val Acc: {best_val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️  Early stopping at epoch {epoch}.")
        break

    print("-" * 70)

print(f"\n✅ QNI-CCP training complete. Best Val Acc: {best_val_acc:.4f}")

# ── Final evaluation ──────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL EVALUATION — Virus-MNIST QNI-CCP MODEL")
print("=" * 70)

ckpt = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
centroids_final = compute_class_centroids(model, train_loader, device, num_classes)

_, clean_acc = evaluate_clean(model, test_loader, loss_fn, device)
qni_acc      = evaluate_qni_robustness(model, test_loader, device, centroids_final)

print(f"\n  Clean Test Accuracy        : {clean_acc:.4f}  ({clean_acc*100:.1f}%)")
print(f"  Under QNI-CCP Perturbation : {qni_acc:.4f}  ({qni_acc*100:.1f}%)")
print(f"  Robustness drop            : {(clean_acc - qni_acc)*100:.1f}%")
print(f"""
COMPARISON TABLE:
  Stage 1 — Clean baseline  ({CLEAN_CHECKPOINT})
  Stage 2 — QNI-CCP only   ({SAVE_PATH})

  Expected: QNI-CCP trained model shows smaller robustness drop
  than clean baseline when attacked at test time.
""")

Using device: cuda

  VIRUS-MNIST QNI-CCP CONFIG
  ε_q (formula / paper ref) : 0.020408
  ε_q (manual override)     : 1.5  ← active
  Ratio (manual / formula)  : 73.5×  stronger
  Save path                 : QNI3_eps1p5.pth

Datasets loaded: 43585 train | 4837 val | 3458 test
Class weights computed: [2.069 0.674 1.71  2.173 6.554 0.78  0.337 0.691 2.019 1.555]
Loaded checkpoint : hybrid_resnet_NOGAN.pth
Previous Val Acc  : 0.9177175935497209
Trainable parameters: 184,234

Computing initial class centroids...


Centroids shape: torch.Size([10, 16])

Starting QNI-CCP Training for 40 epochs
Loss weights : clean=0.625 | QNI=0.25 | centroid=0.125
ε_q (active) : 1.5  |  Focal gamma: 2.0
Save path    : QNI3_eps1p5.pth


Epoch [01/40] | ε_q=1.5 | LR: 0.000200
  Train Loss: 0.4300 | Train Acc: 0.9059
  Val   Loss: 0.4471  | Val   Acc: 0.9237
  💾 Best model saved → QNI3_eps1p5.pth  (Val Acc: 0.9237)
----------------------------------------------------------------------


Epoch [02/40] | ε_q=1.5 | LR: 0.000200
  Train Loss: 0.4240 | Train Acc: 0.9092
  Val   Loss: 0.4428  | Val   Acc: 0.9212
  🕒 No improvement for 1 epoch(s).
----------------------------------------------------------------------


Epoch [03/40] | ε_q=1.5 | LR: 0.000200
  Train Loss: 0.4193 | Train Acc: 0.9108
  Val   Loss: 0.4481  | Val   Acc: 0.9202
  🕒 No improvement for 2 epoch(s).
----------------------------------------------------------------------


...

Epoch [06/40] | ε_q=1.5 | LR: 0.000200
  Train Loss: 0.4123 | Train Acc: 0.9134
  Val   Loss: 0.4517  | Val   Acc: 0.9198
  🕒 No improvement for 5 epoch(s).
----------------------------------------------------------------------


Epoch [07/40] | ε_q=1.5 | LR: 0.000200
  Train Loss: 0.4087 | Train Acc: 0.9145
  Val   Loss: 0.4515  | Val   Acc: 0.9231
  🕒 No improvement for 6 epoch(s).
----------------------------------------------------------------------


Epoch [08/40] | ε_q=1.5 | LR: 0.000100
  Train Loss: 0.4070 | Train Acc: 0.9160
  Val   Loss: 0.4468  | Val   Acc: 0.9227
  🕒 No improvement for 7 epoch(s).
----------------------------------------------------------------------


Epoch [09/40] | ε_q=1.5 | LR: 0.000100
  Train Loss: 0.4019 | Train Acc: 0.9185
  Val   Loss: 0.4488  | Val   Acc: 0.9175
  🕒 No improvement for 8 epoch(s).
----------------------------------------------------------------------
  🔄 Recomputing centroids at epoch 10...


QNI-CCP Training:  11%|█         | 145/1362 [01:05<09:08,  2.22it/s]     